# 15 Generated Sample Grid — Baseline vs SDD

Generate samples from both models with the same random seed and compare them visually.
Train the checkpoints first (notebooks 02, 03).

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU Auto-Detection ───────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"Detected number of GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (No GPU — running on CPU)")


In [ ]:
from pathlib import Path
from src.experiments import (
    load_cfg, deep_update,
    build_trainer_from_checkpoint,
    generate_sample_grid, generate_comparison_grid,
)

cfg_baseline = load_cfg("configs/cifar10.yaml")
cfg_baseline = deep_update(cfg_baseline, {
    "run_name": "cifar10_baseline_mse_only",
    "sdd": {"enabled": False, "lambda_distill": 0.0,
            "use_centering": False, "use_sharpening": False,
            "use_gating": False, "use_projection_head": False},
})
cfg_sdd = load_cfg("configs/cifar10.yaml")
cfg_sdd = deep_update(cfg_sdd, {"run_name": "cifar10_full_sdd"})

ckpt_b   = "outputs/checkpoints/cifar10_baseline_mse_only_last.pt"
ckpt_sdd = "outputs/checkpoints/cifar10_full_sdd_last.pt"

trainer_b   = build_trainer_from_checkpoint(cfg_baseline, ckpt_b)
trainer_sdd = build_trainer_from_checkpoint(cfg_sdd,      ckpt_sdd)
print("✓ Checkpoints loaded")


In [ ]:
from pathlib import Path

Path("outputs/samples").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)

# Individual 8×8 grids
generate_sample_grid(trainer_b,   cfg_baseline, n=64, seed=42,
                     save_path="outputs/samples/baseline_grid.png")
generate_sample_grid(trainer_sdd, cfg_sdd,      n=64, seed=42,
                     save_path="outputs/samples/sdd_grid.png")

# Side-by-side comparison grid (8 samples per model, same seed)
comparison_path = generate_comparison_grid(
    trainers={"Baseline (MSE only)": trainer_b, "Full SDD": trainer_sdd},
    cfgs=    {"Baseline (MSE only)": cfg_baseline, "Full SDD": cfg_sdd},
    n_per_variant=8,
    seed=42,
    save_path="outputs/samples/comparison_grid.png",
)
print(f"✓ Saved: {comparison_path}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, path, title in zip(
    axes,
    ["outputs/samples/baseline_grid.png", "outputs/samples/sdd_grid.png"],
    ["Baseline (MSE only)",                "Full SDD"],
):
    if Path(path).exists():
        ax.imshow(mpimg.imread(path))
        ax.set_title(title, fontsize=13)
    ax.axis("off")
plt.suptitle("Generated samples — seed=42", fontsize=14)
plt.tight_layout()
plt.show()
